# Classifying texts

The `Classification` module contains classes and methods that allows an user to perform classification on textual data by creating, training, and using machine learning models.

The module contains 2 core components:

1. `Classifier`: The core classifier class which dictates the flow of all machine learning models, always abiding the flow of preprocessing the data, initializing the desired model, training it, and evaluating the results
1. `Pipeline`: Abstract class from which all models must inherit from. Children class must define how a model will execute its trainign stage and how it will extract its own features

In this tutorial, we will understand how to use setup and create a Classifier instance to perform a classification experiment on real data. We will first load and process our data, then extract features from it, to then setup a specific `Pipeline` stragegy to be given to our `Classifier` instance

# Loading and Preparing Your Documents

Lets use the `Loader` class from Lexos to import our data contained in the sub-folder `fed_papers`, so we can then use `Tokenizer` and `Scrubber` to preprocess the data

In [1]:
from lexos.io.loader import Loader

loader = Loader()
loader.reset() # Good practice in case there is previous loaded data

loader.load("fed_papers") # Assuming you are running this notebook from lexos/doc_src/docs/tutorials/classification

# Let's check for any errors and what files did we load
print("Errors:", loader.errors)
print("Loaded Names:", loader.names)

SEED = 42 # For reproducing this experiment

from lexos.scrubber.scrubber import Scrubber

scrubber = Scrubber()
scrubber.add_pipe("lower_case") # Simple scrubbing as to maintain feature extraction as ample as possible

# Now let's organize our data into training and test sets
df_all = loader.df
df_train = df_all[df_all["name"].str.contains(r"(?:_H|_M)$", case=False, regex=True, na=False)].copy()
df_train["label"] = df_train["name"].apply(
    lambda name: "HAMILTON" if name.upper().endswith("_H") else "MADISON"
 )
df_test = df_all[df_all["name"].str.contains(r"(?:_D|_C)$", case=False, regex=True, na=False)].copy()

train_texts = df_train["text"].tolist()
train_labels = df_train["label"].tolist()
train_ids = df_train["name"].tolist()
test_texts = df_test["text"].tolist()
test_ids = df_test["name"].tolist()

# A final check to see our trainign and test data sets
print(f"Training docs extracted from Loader: {len(train_texts)}")
print(f"Test docs extracted from Loader: {len(test_texts)}")

Errors: []
Loaded Names: ['FED_18_C', 'FED_19_C', 'FED_20_C', 'FED_49_D', 'FED_50_D', 'FED_51_D', 'FED_52_D', 'FED_53_D', 'FED_54_D', 'FED_55_D', 'FED_56_D', 'FED_57_D', 'FED_58_D', 'FED_62_D', 'FED_63_D', 'FED_11_H', 'FED_12_H', 'FED_13_H', 'FED_15_H', 'FED_16_H', 'FED_17_H', 'FED_1_H', 'FED_21_H', 'FED_22_H', 'FED_23_H', 'FED_24_H', 'FED_25_H', 'FED_26_H', 'FED_27_H', 'FED_28_H', 'FED_29_H', 'FED_30_H', 'FED_31_H', 'FED_32_H', 'FED_33_H', 'FED_34_H', 'FED_35_H', 'FED_36_H', 'FED_59_H', 'FED_60_H', 'FED_61_H', 'FED_65_H', 'FED_66_H', 'FED_67_H', 'FED_68_H', 'FED_69_H', 'FED_6_H', 'FED_70_H', 'FED_71_H', 'FED_72_H', 'FED_73_H', 'FED_74_H', 'FED_75_H', 'FED_76_H', 'FED_77_H', 'FED_78_H', 'FED_79_H', 'FED_7_H', 'FED_80_H', 'FED_81_H', 'FED_82_H', 'FED_83_H', 'FED_84_H', 'FED_85_H', 'FED_8_H', 'FED_9_H', 'FED_2_J', 'FED_3_J', 'FED_4_J', 'FED_5_J', 'FED_64_J', 'FED_10_M', 'FED_14_M', 'FED_37_M', 'FED_38_M', 'FED_39_M', 'FED_40_M', 'FED_41_M', 'FED_42_M', 'FED_43_M', 'FED_44_M', 'FED_45_M',

# Extracting features using `CorpusStats`

Now that we have our train and test data sorted, let's extract some features from it using the `CorpusStats` class

In [2]:
from lexos.corpus import CorpusStats

# CorpusStats requires a list of tuples for its input, so let's create one
corpus_docs = [
    (doc_id, label, text)
    for doc_id, label, text in zip(train_ids, train_labels, train_texts)
]

# Now lets create an instance of CorpusStats and check out the available features for our data
corpus_stats = CorpusStats(docs=corpus_docs)
# corpus_stats.doc_stats_df
my_features = list(corpus_stats.doc_stats_df.columns.values)
print(my_features)
print("Number of extracted features:", len(my_features))

/home/mango/Lexos_Independant_Research/lexos/src/lexos/corpus/corpus_stats.py:404: UserWarning: Loaded spaCy model 'sent_ud_sm' does not include a tagger; skipping syllable-based features.
  warnings.warn(


['total_tokens', 'unique_word_count', 'character_count', 'total_terms', 'punc_count', 'stop_word_count', 'question_count', 'exclamation_count', 'participle_count', 'hapax_legomena', 'hapax_dislegomena', 'sentence_count', 'noun_count', 'verb_count', 'adverb_count', 'num_count', 'adj_count', 'adp_count', 'aux_count', 'cconj_count', 'det_count', 'intj_count', 'part_count', 'pron_count', 'propn_count', 'sconj_count', 'sym_count', 'average_word_length', 'ttr', 'root_ttr', 'log_ttr', 'maas', 'guiraud_index', 'yule_k', 'nominal_ratio', 'simple_nominal_ratio', 'hapax_legomenon_rate', 'vocabulary_density', 'average_sentence_length']
Number of extracted features: 39


# Creating a Pipeline

We shall now create an `Pipeline` configuration, which will dictate hwo our future `Classifier` should perform its internal steps of preprocessing data, initializing a model, training itself, and evaluating its results. For our experiment, we will create an `MLPPipeline` strategy, but this could be any other available Pipeline, or even one of your own!

In [3]:
from lexos.classification import MLPPipeline

my_strategy = MLPPipeline(
    seed=SEED,
    min_df=2,
    test_size=0.2,
    cv_splits=5,
    include_bigrams=True,
    use_smote=True,
    mlp_kwargs={
        "hidden_layer_sizes": (64,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-4,
        "learning_rate_init": 1e-3,
        "max_iter": 1000,
    },
)

Now that we have a `Pipeline` configured, let's pass it to a `Classifier` instance

In [4]:
from lexos.classification import Classifier
classifier = Classifier(
    train_data=corpus_stats.doc_stats_df,
    labels=train_labels,
    pipeline=my_strategy,
    features=my_features
)

classifier.fit()



In [5]:
classifier.metrics

{'accuracy': 0.47692307692307695,
 'balanced_accuracy': 0.5889355742296919,
 'macro_f1': 0.4666988416988417}

In [6]:
classifier.report

,precision,recall,f1-score,support
HAMILTON,0.869565,0.392157,0.540541,51.000000
MADISON,0.261905,0.785714,0.392857,14.000000
accuracy,0.476923,0.476923,0.476923,0.476923
macro avg,0.565735,0.588936,0.466699,65.000000
weighted avg,0.738685,0.476923,0.508732,65.000000


In [9]:
predictions = classifier.predict(corpus_stats.doc_stats_df)

In [14]:
import pandas as pd

predictions_df = pd.DataFrame(predictions)
predictions_df

,0
0,MADISON
1,MADISON
2,HAMILTON
3,MADISON
4,HAMILTON
...,...
60,MADISON
61,HAMILTON
62,MADISON
63,HAMILTON


In [ ]:
print(pd.Series(predictions, index=train_ids))

FED_11_H     MADISON
FED_12_H     MADISON
FED_13_H    HAMILTON
FED_15_H     MADISON
FED_16_H    HAMILTON
              ...   
FED_44_M     MADISON
FED_45_M    HAMILTON
FED_46_M     MADISON
FED_47_M    HAMILTON
FED_48_M    HAMILTON
Length: 65, dtype: object
